# SMITH-Agent evaluation and probe feasibility

This notebook combines the real five-seed MERFISH evaluation output with the full 12,160-gene three-tool feasibility summary from the Agent workflow.

[Open the editable source notebook on GitHub](https://github.com/fym0503/SMITH/blob/main/docs/source/tutorials/notebooks/agent_section/05_SMITH_Agent_Evaluation_source.ipynb)

## Provenance

The code cell verifies the SHA-256 of every bundled result table. These files are copied from completed paper-workspace runs; they are not synthetic replacements for the manuscript outputs.

In [ ]:
from pathlib import Path
import hashlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

def find_repository(start):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "reproducibility").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside a SMITH repository checkout.")

ROOT = find_repository(Path.cwd().resolve())
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 40)

def verify_fixture(relative_path, expected_sha256):
    path = ROOT / "reproducibility" / relative_path
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    assert digest == expected_sha256, f"Checksum mismatch for {path}: {digest}"
    print(f"Verified {relative_path} ({digest[:12]}...)")
    return path


## Analysis

In [ ]:
metrics_path = verify_fixture("fixtures/agent_multi_reference_metrics.tsv", "113eb3ca6329a3aaadb9c0612b1cf25a8c8da87a0778dc5d687f7161558912ab")
feasibility_path = verify_fixture("fixtures/agent_full_tool_pass_summary.tsv", "38bc0c51d8650037fe90e74acb13938265b0b71600cc4711d2361f84d55ed90c")
metrics = pd.read_csv(metrics_path, sep="\t")
accuracy = metrics[metrics["metric"].eq("cell_type_accuracy")]
display(accuracy.groupby(["panel_size", "panel"], as_index=False)["value"].agg(["mean", "std"]).reset_index())
gates = pd.read_csv(feasibility_path, sep="\t")
gates["pass_rate"] = gates["pass_count"] / gates["total_count"]
display(gates)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for panel, group in accuracy.groupby("panel"):
    grouped = group.groupby("panel_size")["value"].agg(["mean", "std"])
    axes[0].errorbar(grouped.index, grouped["mean"], yerr=grouped["std"], marker="o", capsize=3, label=panel)
axes[0].set(xlabel="Panel size", ylabel="Cell-type accuracy", title="Five-seed MERFISH evaluation"); axes[0].legend(frameon=False)
axes[1].bar(gates["gate"], gates["pass_rate"], color="#2f6690")
axes[1].set_ylim(0, 1.05); axes[1].set_ylabel("Fraction of 12,160 genes passing"); axes[1].tick_params(axis="x", rotation=60)
fig.tight_layout(); plt.show()


## Scope

These are completed Agent outputs. Full Figure 6 regeneration still needs reference retrieval, locked test objects and the external ODT/OligoMiner/ProbeDealer resources. The source scripts are archived under `reproducibility/workflows/agent/`.